In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import markdown, numpy, pandas, plotly, nbformat
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install markdown numpy pandas plotly nbformat

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Інтеграція й оптимізація комунікації в складній розподіленій системі</h1>
    <h3>Платформа бронювання авіаквитків «2BeFlüge»</h3>
    <br>
</div>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Communication Strategy |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує високорівневу архітектуру (HLD) та стратегію комунікації для платформи бронювання авіаквитків «2BeFlüge». Система спроєктована для витримки екстремальних навантажень під час флеш-розпродажів (Flash Sales) та базується на мікросервісній архітектурі. Ключовим архітектурним рішенням є гібридна комунікація: використання **gRPC** для синхронного ядра (критичні до затримки транзакції) та подійно-орієнтованої моделі (Event-Driven) на базі **Apache Kafka** для асинхронних підсистем (сповіщення, аналітика, ML-рекомендації).

### **Глобальна архітектурна схема (Високорівнева):**

```mermaid
flowchart TD
    Client([Web / Mobile<br/>Клієнти 2BeFlüge])
    Gateway{{API Gateway / GraphQL}}

    Client -->|API запити| Gateway

    subgraph Core ["Синхронне&nbsp;ядро&nbsp;(Low&nbsp;Latency)"]
        Booking[Сервіс бронювання]
        Inventory[Сервіс інвентарю]
        Payment[Платіжний сервіс]
    end

    Gateway -->|GraphQL / REST| Booking
    Booking <-->|gRPC| Inventory
    Booking <-->|gRPC| Payment
    Gateway -->|REST| Payment

    Broker[[Apache Kafka Broker]]

    Booking -.->|Event: BookingCreated| Broker
    Payment -.->|Event: PaymentProcessed| Broker

    subgraph Async ["Асинхронні&nbsp;підсистеми&nbsp;(Event&nbsp;Driven)"]
        Notification[Сервіс сповіщень]
        Analytics[Сервіс аналітики]
        Recommendation[Сервіс ML рекомендацій]
    end

    Broker -.->|Pub/Sub| Notification
    Broker -.->|Pub/Sub| Analytics
    Broker -.->|Pub/Sub| Recommendation

    classDef sync fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000
    classDef async fill:#fff8e1,stroke:#ff8f00,stroke-width:2px,stroke-dasharray: 5 5,color:#000
    classDef broker fill:#eceff1,stroke:#37474f,stroke-width:3px,color:#000

    class Booking,Inventory,Payment sync
    class Notification,Analytics,Recommendation async
    class Broker broker

    style Core fill:#000000,stroke:#1565c0,stroke-width:1px
    style Async fill:#000000,stroke:#ff8f00,stroke-width:1px
```